In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


## Данные

GPT-4o размечал новости по шкале от −1 до +1 с учётом финансового контекста.
Конвертируем в 3 класса: **NEGATIVE** (score ≤ −0.1), **NEUTRAL** (−0.1 < score < 0.1), **POSITIVE** (score ≥ 0.1).


In [ ]:
news_old = pd.read_parquet(Path('news_descriptions/news_collection_old.parquet'))
GPT4o_df = pd.read_json(Path("news_descriptions/news_descriptions_GPT4o.json")).transpose()

labeled = pd.concat([news_old, GPT4o_df], axis=1)[:15000]
labeled = labeled[labeled["body"].astype(str) != '""'].reset_index(drop=True)
labeled["sentiment_score"] = pd.to_numeric(labeled["sentiment_score"], errors="coerce")
labeled = labeled.dropna(subset=["sentiment_score"]).reset_index(drop=True)

def score_to_label(s):
    if s >= 0.1:  return 2  # POSITIVE
    if s <= -0.1: return 0  # NEGATIVE
    return 1                # NEUTRAL

labeled["label"] = labeled["sentiment_score"].apply(score_to_label)
print(labeled["label"].map({0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}).value_counts())


In [ ]:
counts = labeled["label"].value_counts().sort_index()
plt.figure(figsize=(5, 3))
plt.bar(["NEGATIVE", "NEUTRAL", "POSITIVE"], counts.values,
        color=["tomato", "steelblue", "seagreen"])
plt.title("Распределение классов (GPT-4o разметка)")
plt.ylabel("Новостей")
plt.tight_layout()
plt.show()


## Fine-Tuning

Базовая модель: `cointegrated/rubert-tiny2`. 85% данных — обучение, 15% — валидация. 3 эпохи.


In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LEN    = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

rng = np.random.default_rng(42)
idx = rng.permutation(len(labeled))
n_val = int(0.15 * len(labeled))
val_idx, train_idx = idx[:n_val], idx[n_val:]

train_df = labeled.iloc[train_idx].reset_index(drop=True)
val_df   = labeled.iloc[val_idx].reset_index(drop=True)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")


In [ ]:
class FinSentDataset(Dataset):
    def __init__(self, texts, labels):
        enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.enc    = enc
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.enc.items()}, self.labels[idx]

train_ds = FinSentDataset(
    train_df["body"].fillna("").str.strip('"').tolist(),
    train_df["label"].tolist(),
)
val_ds = FinSentDataset(
    val_df["body"].fillna("").str.strip('"').tolist(),
    val_df["label"].tolist(),
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

ID2LABEL = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss, steps = 0.0, 0
    for batch_enc, batch_labels in train_loader:
        batch_enc    = {k: v.to(device) for k, v in batch_enc.items()}
        batch_labels = batch_labels.to(device)
        optimizer.zero_grad()
        loss = model(**batch_enc, labels=batch_labels).loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps += 1

    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_enc, batch_labels in val_loader:
            batch_enc = {k: v.to(device) for k, v in batch_enc.items()}
            preds = model(**batch_enc).logits.argmax(dim=-1).cpu()
            correct += (preds == batch_labels).sum().item()

    print(f"Epoch {epoch + 1}/{EPOCHS}  loss={total_loss / steps:.4f}  val_acc={correct / len(val_ds):.3f}")


### Метрики на валидации


In [ ]:
model.eval()
all_preds = []
with torch.no_grad():
    for batch_enc, _ in val_loader:
        batch_enc = {k: v.to(device) for k, v in batch_enc.items()}
        all_preds.extend(model(**batch_enc).logits.argmax(dim=-1).cpu().tolist())

y_val = val_df["label"].tolist()
for cls, name in ID2LABEL.items():
    tp = sum(1 for p, t in zip(all_preds, y_val) if p == cls and t == cls)
    fp = sum(1 for p, t in zip(all_preds, y_val) if p == cls and t != cls)
    fn = sum(1 for p, t in zip(all_preds, y_val) if p != cls and t == cls)
    prec = tp / (tp + fp) if tp + fp else 0
    rec  = tp / (tp + fn) if tp + fn else 0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0
    print(f"{name:10s}  precision={prec:.3f}  recall={rec:.3f}  f1={f1:.3f}")


## Сохранение модели

Модель и токенизатор сохраняются в `models/rubert-financial-sentiment/`.
При следующем запуске блока инференса — грузятся оттуда, обучение не нужно.


In [ ]:
SAVE_DIR = Path("models/rubert-financial-sentiment")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved → {SAVE_DIR}")
